# 🕷️ Recolección de datos via Web Scraping

## ¿Por qué scraping y no una API?
MercadoLibre tiene una API pública, pero tiene límites de requests y no devuelve
todos los campos que necesitamos (año, km, moneda) de forma limpia y consistente.
El scraping nos da control total sobre qué extraemos y cómo lo estructuramos.

## ¿Por qué Playwright y no BeautifulSoup directamente?
MercadoLibre carga su contenido con JavaScript del lado del cliente. BeautifulSoup
solo lee HTML estático — si le mandamos el HTML crudo de ML, recibe una página vacía.
Playwright levanta un browser real (Chromium) que ejecuta el JavaScript y espera
a que el contenido esté renderizado antes de extraerlo.

## Técnicas anti-bot implementadas
MercadoLibre detecta scrapers y los bloquea. Para evitarlo implementé:
- **User-Agent real:** el browser se identifica como Chrome 124 en Windows
- **Viewport humano:** resolución 1280x800, no el default de Playwright
- **WebDriver oculto:** se elimina la propiedad `navigator.webdriver` que los
  sitios usan para detectar automatización
- **Sleep random:** entre 2 y 4 segundos entre requests para imitar comportamiento humano
- **Locale argentino:** `es-AR` para recibir precios en formato local

## Output
El scraper genera `data/raw/precios_autos_raw.csv` con 3.780 registros y 7 columnas:
`modelo`, `titulo`, `precio`, `moneda`, `año`, `km`, `ubicacion`.

La columna `moneda` es clave — MercadoLibre permite publicar en USD o ARS,
y capturar el símbolo de moneda junto al precio permite una conversión correcta
en la fase de limpieza.

## Código del scraper

El scraper está implementado en `src/scraper.py` y se ejecuta desde la terminal:

```bash
python src/scraper.py
```

**¿Por qué no se ejecuta desde el notebook?**
Playwright usa su propio event loop que es incompatible con el event loop
de Jupyter. Esta es una limitación conocida — los scrapers con Playwright
siempre se corren como scripts independientes, no dentro de notebooks.

## Output generado

El scraper genera `data/raw/precios_autos_raw.csv`. A continuación
se muestra una muestra del dataset crudo resultante:

In [2]:
import pandas as pd

df_raw = pd.read_csv("../data/raw/precios_autos_raw.csv")
print(f"Shape: {df_raw.shape}")
print(f"\nDistribución por modelo:\n{df_raw['modelo'].value_counts()}")
df_raw.head(10)

Shape: (3780, 7)

Distribución por modelo:
modelo
Toyota Etios           240
Toyota Corolla         240
Toyota Yaris           240
Chevrolet Prisma       240
Fiat Cronos            240
Renault Logan          240
VW Virtus              240
VW Voyage              240
Nissan Versa           240
Fiat Siena             240
VW Polo                240
VW Gol                 240
Ford Fiesta            240
Ford Ka                240
Peugeot 208            240
Chevrolet Onix Plus    180
Name: count, dtype: int64


,modelo,titulo,precio,moneda,año,km,ubicacion
0,Toyota Etios,Toyota Etios 1.5 Sedan Xls,13200.0,US$,2018,89.000 Km,Capital Federal - Capital Federal
1,Toyota Etios,Toyota Etios 1.5 Sedan Xls,18100000.0,$,2018,133.000 Km,Capital Federal - Capital Federal
2,Toyota Etios,Toyota Etios 1.5 Sedan Xls,11500000.0,$,2016,101.000 Km,Avellaneda - Bs.As. G.B.A. Sur
3,Toyota Etios,Toyota Etios 1.5 Xls,17000.0,US$,2021,80.000 Km,San Miguel - Bs.As. G.B.A. Norte
4,Toyota Etios,Toyota Etios 1.5 5 Ptas Xls 4at,18000000.0,$,2017,47.000 Km,Vicente López - Bs.As. G.B.A. Norte
5,Toyota Etios,Toyota Etios 1.5 Sedan Xls,23499000.0,$,2021,30.000 Km,Quilmes - Bs.As. G.B.A. Sur
6,Toyota Etios,Toyota Etios 1.5 Sedan Xls,21000000.0,$,2019,55.600 Km,Capital Federal - Capital Federal
7,Toyota Etios,Toyota Etios 1.5 Xs,14500000.0,$,2016,150.000 Km,Vicente López - Bs.As. G.B.A. Norte
8,Toyota Etios,Toyota Etios 1.5 Platinum At,12500.0,US$,2017,148.000 Km,Tres De Febrero - Bs.As. G.B.A. Oeste
9,Toyota Etios,Toyota Etios 1.5 Platinum At,11999.0,US$,2018,153.200 Km,Capital Federal - Capital Federal


## Resultado del scraping

| Métrica | Valor |
|---|---|
| Modelos scrapeados | 16 |
| Páginas por modelo | 5 |
| Publicaciones por página | ~48 |
| **Total de registros** | **3.780** |

### Decisiones de diseño
- **5 páginas por modelo:** cada página trae ~48 publicaciones, generando ~240 registros
  por modelo. Es suficiente para una distribución estadísticamente representativa
  sin tardar horas en ejecutarse.
- **`headless=False`:** el browser se muestra visible durante el scraping porque
  MercadoLibre bloquea con más agresividad los browsers invisibles.
- **Precio y moneda separados:** en lugar de limpiar la moneda en el scraper,
  la guardamos como columna separada para mantener los datos crudos sin transformar.
  La conversión se hace en la fase de limpieza con el tipo de cambio del día.